## Architecture Setup

This notebook uses pretrained BERT components to demonstrate token embeddings and a separate custom causal attention layer. It is a forward-pass walkthrough, not a complete autoregressive model training pipeline.

- **Vocabulary size:** `30,522`, matching the `bert-base-uncased` tokenizer and pretrained embedding table.
- **Hidden size:** `768`.
- **Input token IDs:** `(B, 8)` for a batch of sequences padded or truncated to 8 tokens.
- **Custom attention input/output:** `(B, 8, 768)`.
- **BERT masked-LM reference logits:** `(B, 8, 30,522)`, produced by BERT's pretrained prediction head for shape reference only; they are separate from the custom attention output.
- **Causal masking:** enabled for the BERT reference forward pass with `is_decoder=True` and separately in the custom attention layer. This changes the attention mask used at inference time; it does not change BERT's original bidirectional pretraining.

## Step 1: Tokenization

Start with an example input sequence. This is a demonstration input, not a training example for updating model weights.

- **Example input:** `"The bank of the river flooded"`

In [ ]:
import torch
from transformers import AutoTokenizer, BertConfig, BertLMHeadModel

# Define an example input sequence
sentence = "The bank of the river flooded"

# Load the tokenizer and pretrained BERT model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
config = BertConfig.from_pretrained("bert-base-uncased")
config.is_decoder = True
model = BertLMHeadModel.from_pretrained("bert-base-uncased", config=config)
model.eval()

# Use a fixed sequence length so the shape walkthrough is reproducible
inputs = tokenizer(
    sentence,
    max_length=8,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)
print("Input IDs shape:", inputs["input_ids"].shape)
print("Attention mask:", inputs["attention_mask"])

## Step 2: Get Token Embeddings

For the custom transformer path, start with vectors from BERT's pretrained word-embedding table. These are token embeddings only: they have not yet passed through BERT's positional embeddings or transformer layers. We will add positional information once, then apply our custom layer normalization and attention.

In [ ]:
# Run BERT's pretrained LM head as a reference only.
# This output/loss path is separate from the custom attention layer below.
labels = inputs["input_ids"].clone()
labels.masked_fill_(inputs["attention_mask"] == 0, -100)

with torch.no_grad():
    bert_outputs = model(
        **inputs,
        return_dict=True,
    )

bert_logits = bert_outputs.logits
print("BERT reference logits shape:", bert_logits.shape)

# Start the custom path from pretrained word embeddings, before BERT adds
# positional embeddings or runs its transformer layers.
x_embeddings = model.get_input_embeddings()(inputs["input_ids"])
print("Raw word embedding shape:", x_embeddings.shape)

## Step 3: Add Positional Encodings

Self-attention does not inherently encode token order. We add sinusoidal positional encodings to the raw word embeddings on the custom path. Unlike BERT's final hidden states, these word embeddings do not already contain BERT's positional or contextual information, so position is added only once here.

In [9]:
import numpy as np

def get_positional_encoding(seq_len: int, d_model: int) -> torch.Tensor:
    """Generates a sinusoidal positional encoding tensor."""
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe.unsqueeze(0)  # Add batch dimension shape: [1, seq_len, d_model]

# Generate positional encodings matched to our sequence length (8) and embedding dimension (768)
B, T, D = x_embeddings.shape
pos_encoding = get_positional_encoding(T, D).to(x_embeddings.device)

# Combine the embeddings and positional encodings
x_with_pos = x_embeddings + pos_encoding

print(f"Positional Encoding Shape:  {pos_encoding.shape}")
print(f"Position-Aware Input Shape: {x_with_pos.shape}")

Positional Encoding Shape:  torch.Size([1, 8, 768])
Position-Aware Input Shape: torch.Size([1, 8, 768])


## Step 4: Layer Normalization

Layer Normalization computes statistics across the feature dimension of each token independently. Unlike Batch Normalization, it does not couple a token's normalized values to other examples or sequence positions, which works naturally with variable-length sequences.

In this focused pre-norm-style example, we normalize each position-aware token vector before custom self-attention. A full Transformer block would also include a residual connection and a feed-forward sublayer; those are outside this notebook's current scope.

In [10]:
import torch
import torch.nn as nn

class CustomLayerNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        # Learnable parameters: gamma (initialized to 1), beta (initialized to 0)
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, unbiased=False, keepdim=True)  # MLE variance
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

# Initialize LayerNorm with BERT's hidden dimension size (768)
layer_norm = CustomLayerNorm(normalized_shape=768).to(x_with_pos.device)

# Apply Layer Normalization to the position-aware embeddings
x_norm = layer_norm(x_with_pos)

print(f"Original Embedding Shape:      {x_with_pos.shape}")
print(f"Layer-Normalized Shape:        {x_norm.shape}")

Original Embedding Shape:      torch.Size([1, 8, 768])
Layer-Normalized Shape:        torch.Size([1, 8, 768])


## Step 5: Custom Multi-Head Self-Attention

We define a custom attention layer with both causal masking and the tokenizer's padding mask. The layer uses `d_model=768` and `num_heads=12`, giving each head a key/value dimension of 64. Its weights are randomly initialized for this forward-pass demonstration; they are not trained in this notebook.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        is_causal: bool = False,
    ) -> torch.Tensor:
        batch_size, seq_len, hidden_size = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q, k, v = (
            tensor.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
            for tensor in (q, k, v)
        )
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)

        if attention_mask is not None:
            if attention_mask.shape != (batch_size, seq_len):
                raise ValueError("attention_mask must have shape (batch_size, seq_len)")
            key_mask = attention_mask[:, None, None, :].to(torch.bool)
            scores = scores.masked_fill(~key_mask, float("-inf"))

        if is_causal:
            causal_mask = torch.tril(
                torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool)
            )
            scores = scores.masked_fill(~causal_mask, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = torch.matmul(attn_weights, v)
        # Combine each (8,64) size matrix to create final context representation of all 12 heads
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, hidden_size)
        return self.out_proj(context)

## Step 6: Verify the Attention Forward Pass

Pass the normalized embeddings and tokenizer attention mask through the custom attention layer with causal masking enabled. The checks below verify that masking is accepted and the output shape is preserved; they do not measure whether the randomly initialized layer has learned useful representations.

In [ ]:
# Instantiate the custom attention layer with BERT-compatible dimensions.
mha_768 = MultiHeadAttention(d_model=768, num_heads=12)

# Pass both the padding mask and causal mask into custom attention.
attention_output = mha_768(
    x_norm,
    attention_mask=inputs["attention_mask"],
    is_causal=True,
)

print("Layer-normalized input shape:", x_norm.shape)
print("Custom attention output shape:", attention_output.shape)

### Shape Verification

Check that custom attention preserves the batch, sequence, and hidden dimensions. Matching shapes confirm dimensional compatibility only; they do not establish that the attention weights are meaningful or trained.

In [ ]:
assert x_norm.shape == attention_output.shape, (
    f"Shape mismatch: {x_norm.shape} != {attention_output.shape}"
)

print("Input shape:         ", tuple(x_norm.shape))
print("Attention output:    ", tuple(attention_output.shape))
print("Shape-preservation check passed; this does not evaluate model quality.")

### Summary of the Demonstration

1. **Tokenization:** Convert the example text into token IDs and an attention mask.
2. **BERT reference pass:** Compute pretrained LM-head logits/loss separately for comparison; these outputs do not train the custom attention layer.
3. **Word embeddings:** Look up pretrained token vectors with shape `(B, sequence_length, 768)` before adding positions or contextual attention.
4. **Positional encoding:** Add sinusoidal position information once to the custom path.
5. **Layer normalization:** Normalize each token vector across the hidden dimension.
6. **Custom attention:** Apply causal masking and the tokenizer padding mask; verify output dimensions are preserved.

This notebook demonstrates a forward pass through a custom attention layer. It does not include an optimizer or train that layer.